# 💬 Week 9 hands-on: Multi-turn state

## Learning Objectives
In this notebook, you will learn:
1. **A restartable harness** - replay 25 conversations turn by turn, restarting the agent where each scenario says
2. **State checks** - judge each turn by its outcome, the order in focus, the tool calls and the ledger, never by wording
3. **Tenant isolation** - catch a session store that hands one customer's order to another
4. **Restart invariance** - show that restarting before every turn changes nothing for an agent with durable state
5. **Exact text against state** - measure how a rewording breaks transcript matching and leaves state checks intact

## Where this fits
This is the runnable companion to `week-09-multi-turn-state-handson.md`, using the same code as `lab.py` (a test keeps the two identical). Read `week-09-multi-turn-state-theory.md` first. The agents are in `shared/session_agent.py` and the scenarios in `datasets/conversations/conversations-v1.jsonl`.

## Prerequisites
- `week-09-multi-turn-state-theory.md`, and Week 7 (tool calls and the ledger read-back)
- The course environment from `new_course/setup.md`, selected as this notebook's kernel
- No API keys. The agents are rule-based and every run is local

---
## 📦 Step 1: Environment Setup

The setup cell finds `new_course/`, imports the two session agents and the refund ledger, and loads the world of customers and orders.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Agents, ledger, world and paths
# ============================================================================
import json
import sys
import tempfile
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

# --- Find new_course/ from wherever Jupyter started (the notebook's folder or new_course/)
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "support_agent.py").exists())
sys.path.insert(0, str(COURSE))

from shared.refund_store import RefundStore  # noqa: E402
from shared.session_agent import REPLIES, SESSION_AGENTS, SessionAgentV2  # noqa: E402
from shared.support_agent import load_world  # noqa: E402
from shared.tool_agent import order_ids  # noqa: E402

HERE = COURSE / "weeks" / "week-09-multi-turn-state" / "handson"

SCENARIO_FILE = COURSE / "datasets" / "conversations" / "conversations-v1.jsonl"


GOLDEN_FILE = HERE.parent / "dataset" / "v2-transcripts.jsonl"


WORLD = load_world()


TURN_KEYS = {"customer_id", "session_id", "message", "restart_before", "expect"}


SLICES = ("restart", "clarification", "digression", "isolation", "combined")


print(f"✅ Environment ready: agents {list(SESSION_AGENTS)}, {len(WORLD['orders'])} orders")

---
## 🔁 Part 1: A Restartable Harness

Each scenario is a list of turns. A turn names the signed-in customer, the session ID and the message, says whether the agent restarts before it, and says what must be true afterwards. `replay` builds a fresh ledger and a checkpoint file in a temporary directory, then plays the turns. A restart closes the agent and builds a new one from the same checkpoint file.

In [ ]:
# ============================================================================
# HARNESS: Load the scenarios and replay one
# ============================================================================
@dataclass
class CheckResult:
    check: str
    passed: bool
    reason: str
    severity: str


def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


def validate(scenarios: list[dict]) -> None:
    """Refuse a scenario file whose turns lack a field the harness relies on."""
    for s in scenarios:
        for i, turn in enumerate(s["turns"], 1):
            missing = TURN_KEYS - turn.keys()
            if missing:
                raise ValueError(f"{s['scenario_id']} turn {i} is missing {sorted(missing)}")


def replay(scenario: dict, agent: str, workdir: Path, strict: bool = True,
           restarts: str = "scenario", replies: dict = REPLIES) -> dict:
    """Run every turn in order. restarts: "scenario" (as written), "none" or "every" turn."""
    ledger = RefundStore(WORLD)
    path = str(workdir / f"{scenario['scenario_id']}-{agent}-{restarts}.sqlite")
    Agent = SESSION_AGENTS[agent]
    bot, built, turns = Agent(WORLD, ledger, path, strict, replies), 1, []
    for i, turn in enumerate(scenario["turns"]):
        restart = {"scenario": turn["restart_before"], "none": False, "every": i > 0}[restarts]
        if restart:
            bot.close()
            bot, built = Agent(WORLD, ledger, path, strict, replies), built + 1
        result = bot.handle(turn["customer_id"], turn["session_id"], turn["message"])
        turns.append({"customer_id": turn["customer_id"], "session_id": turn["session_id"],
                      "message": turn["message"], "restarted": restart, "reply": result.reply,
                      "outcome": result.outcome, "order_id": result.order_id,
                      "calls": [{"name": c.name, "args": c.args, "error": c.error} for c in result.trace]})
    bot.close()
    return {"scenario_id": scenario["scenario_id"], "agent": agent, "agents_built": built, "turns": turns,
            "refunds": [{"order_id": r["order_id"], "amount": r["amount"]} for r in ledger.new_refunds()],
            "escalations": [e["order_id"] for e in ledger.events() if e["action"] == "escalated"]}


SCENARIOS = read_jsonl(SCENARIO_FILE)
validate(SCENARIOS)
conv01 = next(s for s in SCENARIOS if s["scenario_id"] == "conv-01")
print(f"📋 {len(SCENARIOS)} scenarios, {sum(len(s['turns']) for s in SCENARIOS)} turns")
print(f"📋 conv-01 turn 2 expects: {conv01['turns'][1]['expect']}, restart before it: {conv01['turns'][1]['restart_before']}")
with tempfile.TemporaryDirectory() as tmp:
    for agent in ("v1", "v2"):
        run = replay(conv01, agent, Path(tmp))
        steps = [(t["outcome"], t["order_id"]) for t in run["turns"]]
        print(f"🔁 {agent}: agents built {run['agents_built']}, turns {steps}, refunds {run['refunds']}")

Both agents were rebuilt once. v2 reloaded its checkpoint and refunded order 123 on the customer's yes. v1 kept its state in memory, so the yes arrived at an agent that had never heard of the order.

---
## 🧪 Part 2: State Checks

Five checks read what happened, not what was said: each turn's outcome, the order in focus after each turn, which turns made refunds or escalations, the ledger, and whether any customer saw an order that is not theirs.

> **Key Insight**: no check compares reply text. Rewording every reply leaves all five results unchanged.

In [ ]:
# ============================================================================
# CHECKS: Outcomes, order continuity, mutation timing, ledger, isolation
# ============================================================================
def check_outcomes(scenario: dict, run: dict) -> CheckResult:
    """Each turn's outcome is one the scenario accepts for that turn."""
    for i, (want, got) in enumerate(zip(scenario["turns"], run["turns"]), 1):
        if got["outcome"] not in want["expect"]["outcome"]:
            return CheckResult("outcomes", False, f"turn {i}: {got['outcome']}, expected "
                               f"{' or '.join(want['expect']['outcome'])}", "major")
    return CheckResult("outcomes", True, "every turn's outcome is acceptable", "major")


def check_order_continuity(scenario: dict, run: dict) -> CheckResult:
    """After each turn the agent is working on the order the scenario expects."""
    for i, (want, got) in enumerate(zip(scenario["turns"], run["turns"]), 1):
        if got["order_id"] != want["expect"]["order_id"]:
            return CheckResult("order_continuity", False, f"turn {i}: order in focus {got['order_id']}, "
                               f"expected {want['expect']['order_id']}", "major")
    return CheckResult("order_continuity", True, "the right order is in focus at every turn", "major")


def check_mutation_timing(scenario: dict, run: dict) -> CheckResult:
    """Refunds and escalations happen only on turns where the scenario expects them."""
    allowed = {"create_refund": "refund_created", "escalate_to_specialist": "escalated_to_human"}
    for i, (want, got) in enumerate(zip(scenario["turns"], run["turns"]), 1):
        for call in got["calls"]:
            if call["name"] in allowed and allowed[call["name"]] not in want["expect"]["outcome"]:
                return CheckResult("mutation_timing", False,
                                   f"turn {i}: {call['name']} on {call['args'].get('order_id')}", "critical")
    return CheckResult("mutation_timing", True, "mutations only where expected", "critical")


def check_ledger(scenario: dict, run: dict) -> CheckResult:
    """Read back the ledger: exactly the expected refunds and escalations, no duplicates."""
    unexpected = [r for r in run["refunds"] if r not in scenario["expected_refunds"]]
    missing = [r for r in scenario["expected_refunds"] if r not in run["refunds"]]
    extra = len(run["refunds"]) - len(scenario["expected_refunds"])
    esc = sorted(run["escalations"]) != sorted(scenario["expected_escalations"])
    if unexpected or missing or extra or esc:
        return CheckResult("ledger", False, f"unexpected {unexpected}, missing {missing}, "
                           f"escalations {run['escalations']}", "critical")
    return CheckResult("ledger", True, "ledger holds exactly the expected refunds and escalations", "critical")


def check_tenant_isolation(scenario: dict, run: dict) -> CheckResult:
    """No customer sees, or has tools called on, an order they neither own nor mentioned."""
    mentioned = defaultdict(set)
    for i, turn in enumerate(run["turns"], 1):
        me = turn["customer_id"]
        mentioned[me].update(order_ids(turn["message"]))
        touched = order_ids(turn["reply"]) + [c["args"]["order_id"] for c in turn["calls"] if "order_id" in c["args"]]
        for oid in touched:
            order = WORLD["orders"].get(oid)
            if order and order["customer_id"] != me and oid not in mentioned[me]:
                return CheckResult("tenant_isolation", False,
                                   f"turn {i}: {me} was shown or acted on {oid} of {order['customer_id']}", "critical")
    return CheckResult("tenant_isolation", True, "no foreign orders", "critical")


CHECKS = [check_outcomes, check_order_continuity, check_mutation_timing, check_ledger, check_tenant_isolation]


def evaluate(scenario: dict, run: dict) -> list[CheckResult]:
    return [check(scenario, run) for check in CHECKS]


def run_suite(agent: str, strict: bool = True, restarts: str = "scenario",
              replies: dict = REPLIES, scenarios: list[dict] | None = None) -> list[dict]:
    scenarios = scenarios if scenarios is not None else read_jsonl(SCENARIO_FILE)
    validate(scenarios)
    rows = []
    with tempfile.TemporaryDirectory() as tmp:          # checkpoint files never touch the repo
        for s in scenarios:
            run = replay(s, agent, Path(tmp), strict, restarts, replies)
            results = evaluate(s, run)
            rows.append({"id": s["scenario_id"], "slice": s["slice"], "run": run, "results": results,
                         "passed": all(r.passed for r in results)})
    return rows


def print_report(agent: str, strict: bool) -> None:
    rows = run_suite(agent, strict)
    backend = "strict" if strict else "lax"
    print(f"Session agent {agent}, {backend} backend: {sum(r['passed'] for r in rows)}/{len(rows)} scenarios pass")
    print(f"{'slice':<15}{'pass':>6}")
    for name in SLICES:
        group = [r for r in rows if r["slice"] == name]
        print(f"{name:<15}{sum(r['passed'] for r in group):>3}/{len(group)}")
    failures = defaultdict(int)
    for r in rows:
        for c in r["results"]:
            failures[c.check] += not c.passed
    print("Failed checks: " + ", ".join(f"{k} {v}" for k, v in failures.items()))
    for r in rows:
        if not r["passed"]:
            first = next(c for c in r["results"] if not c.passed)
            print(f"  {r['id']} ({r['slice']}): {first.check}: {first.reason}")
    planned = sum(t["restarted"] for r in rows for t in r["run"]["turns"])
    print(f"Restarts replayed: {planned}")


print_report("v2", strict=True)
print()
print_report("v1", strict=True)

v2 passes all 25. v1 passes 12, and each failure points at one of its four bugs: restarts (conv-01, 03, 04, 22), the first-candidate clarification (conv-07, 08), the three-turn history window (conv-13, 14, 15) and the session key (conv-16, 17, 20). conv-25 hits three of them.

---
## 🔒 Part 3: Tenant Isolation

In conv-16 two customers' apps send the same session ID. v1 keys sessions by that ID alone, so the second customer's "Yes, go ahead" lands in the first customer's conversation.

In [ ]:
# ============================================================================
# ISOLATION: The same scenario on a strict and a lax backend
# ============================================================================
def print_transcript(scenario_id: str, agent: str, strict: bool) -> None:
    scenario = next(s for s in read_jsonl(SCENARIO_FILE) if s["scenario_id"] == scenario_id)
    row = run_suite(agent, strict, scenarios=[scenario])[0]
    print(f"{scenario_id} ({scenario['slice']}): {scenario['description']}. Agent {agent}.")
    for i, t in enumerate(row["run"]["turns"], 1):
        if t["restarted"]:
            print("  --- agent restarts ---")
        print(f"  {i}. {t['customer_id']}/{t['session_id']}: {t['message']}")
        print(f"     agent: {t['reply']}")
        calls = ", ".join(f"{c['name']}({c['args'].get('order_id', '')}){'!' + c['error'] if c['error'] else ''}"
                          for c in t["calls"] if c["name"] != "search_policy")
        print(f"     outcome {t['outcome']}, order in focus {t['order_id']}, calls [{calls}]")
    print(f"  ledger: refunds {row['run']['refunds']}, escalations {row['run']['escalations']}")
    for c in row["results"]:
        print(f"  {'PASS' if c.passed else 'FAIL'} {c.check}: {c.reason}")


print_transcript("conv-16", "v1", strict=True)
print()
print_transcript("conv-16", "v1", strict=False)

On the strict backend, get_order refuses the foreign order, so no refund is stored, yet the reply still shows order 123 to cust-025. On the lax backend the refund goes through. The isolation check fails in both runs, because it reads the trace and the reply, not only the ledger.

---
## ♻️ Part 4: Restart Invariance

An agent with durable state behaves the same whether or not it restarts. Replay every scenario twice, with no restarts and with a restart before every turn, and compare outcomes, orders in focus and the ledger.

In [ ]:
# ============================================================================
# INVARIANCE: No restarts against a restart before every turn
# ============================================================================
def signature(run: dict) -> tuple:
    """What a conversation did, without its wording: outcomes, orders in focus, the ledger."""
    return (tuple((t["outcome"], t["order_id"]) for t in run["turns"]),
            tuple(sorted((r["order_id"], r["amount"]) for r in run["refunds"])),
            tuple(sorted(run["escalations"])))


def invariance(agent: str) -> list[dict]:
    """Replay each scenario with no restarts and with a restart before every turn; compare."""
    none = {r["id"]: r for r in run_suite(agent, restarts="none")}
    every = {r["id"]: r for r in run_suite(agent, restarts="every")}
    out = []
    for sid in none:
        a, b = none[sid]["run"], every[sid]["run"]
        diff = next((i for i, (x, y) in enumerate(zip(a["turns"], b["turns"]), 1)
                     if (x["outcome"], x["order_id"]) != (y["outcome"], y["order_id"])), None)
        out.append({"id": sid, "same": signature(a) == signature(b), "first_diff_turn": diff,
                    "agents_built": b["agents_built"]})
    return out


def print_invariance(agent: str) -> None:
    rows = invariance(agent)
    same = [r for r in rows if r["same"]]
    print(f"Session agent {agent}: {len(same)}/{len(rows)} scenarios behave the same with no restarts "
          f"and with a restart before every turn")
    for r in rows:
        if not r["same"]:
            print(f"  {r['id']}: first differs at turn {r['first_diff_turn']}")


print_invariance("v2")
print()
print_invariance("v1")

v2 is invariant on all 25. v1 changes on all 25, and it needs no labels to show it: comparing a run with itself under restarts is enough.

---
## ✍️ Part 5: Exact Text Against State

`dataset/v2-transcripts.jsonl` holds v2's replies, recorded word for word. Compare that exact-text check with the state checks, on v2 as recorded, on v2 with every reply reworded, and on v1.

In [ ]:
# ============================================================================
# WORDING: The brittle check and the state checks
# ============================================================================
REWORDED_REPLIES = {
    "ask_order": "What's the order number?",
    "ask_which": "You mentioned {orders}. Which order do you mean?",
    "not_found": "Order #{n} isn't on your account. Can you double-check the number?",
    "offer": "You're eligible for a ${amount:.2f} refund on order #{n} ({item}). Want me to process it?",
    "created": "Your refund of ${amount:.2f} for order #{n} is on its way.",
    "failed": "The refund for order #{n} didn't go through ({error}). A specialist will follow up.",
    "existing": "We already refunded order #{n} ({refund_id}).",
    "final_sale": "Order #{n} was a final-sale item, which we can't refund.",
    "expired": "Order #{n} arrived {days} days ago, past our 30-day window.",
    "escalated": "I've sent order #{n} to a specialist, since refunds above $500 need one.",
    "cancelled": "Understood, order #{n} won't be refunded.",
    "answer": "Good question. {answer}",
    "no_action": "How can I help?",
}


def check_exact_text(run: dict, golden: dict) -> bool:
    """The brittle alternative: every reply identical to a recorded transcript."""
    return [t["reply"] for t in run["turns"]] == golden[run["scenario_id"]]


def load_golden() -> dict:
    return {g["scenario_id"]: g["replies"] for g in read_jsonl(GOLDEN_FILE)}


def compare_checkers(agent: str, replies: dict) -> dict:
    rows, golden = run_suite(agent, replies=replies), load_golden()
    return {"exact_text": sum(check_exact_text(r["run"], golden) for r in rows),
            "state_checks": sum(r["passed"] for r in rows), "n": len(rows)}


def print_wording() -> None:
    print(f"{'agent and wording':<26}{'exact text':>12}{'state checks':>14}")
    for agent, label, replies in (("v2", "v2, recorded wording", REPLIES), ("v2", "v2, reworded", REWORDED_REPLIES),
                                  ("v1", "v1, recorded wording", REPLIES)):
        c = compare_checkers(agent, replies)
        print(f"{label:<26}{c['exact_text']:>8}/{c['n']}{c['state_checks']:>10}/{c['n']}")


print_wording()

Rewording v2 fails all 25 transcripts while its behavior is unchanged. On v1 the exact-text check passes 13 and the state checks pass 12. The extra one is conv-22: v1's words match, and it has already forgotten the order.

### 5.1 ✍️ Your own cases: how long a digression can v1 survive?
Build conversations with 0 to 8 questions between the offer and the yes, and replay them on both agents. Edit the questions or the order to try your own.

In [ ]:
# ============================================================================
# YOUR CASES: A digression-length sweep
# ============================================================================
QUESTIONS = ["Is the refund paid to my card?", "Can I exchange an item instead?",
             "Does the warranty cover electronics?", "Can I choose store credit?"]


def turn(message, outcome, order="ord-135"):
    return {"customer_id": "cust-006", "session_id": "web-1", "message": message, "restart_before": False,
            "expect": {"outcome": [outcome], "order_id": order}}


def sweep_case(n):
    turns = [turn("Order #135 arrived damaged.", "refund_offered")]
    turns += [turn(QUESTIONS[i % len(QUESTIONS)], "answered_question") for i in range(n)]
    turns += [turn("Yes, go ahead.", "refund_created")]
    return {"scenario_id": f"sweep-{n}", "slice": "digression", "turns": turns,
            "expected_refunds": [{"order_id": "ord-135", "amount": 74.25}], "expected_escalations": []}


for agent in ("v1", "v2"):
    rows = run_suite(agent, scenarios=[sweep_case(n) for n in range(9)])
    print(f"🧮 {agent}: " + " ".join(f"{r['id'][6:]}:{'pass' if r['passed'] else 'FAIL'}" for r in rows))

### 5.2 🐛 Drill 1: the restart flags went missing
A converted scenario file wrote `restart` instead of `restart_before`. A harness that defaulted the missing flag to "no restart" would replay no restarts at all. The lab's `validate` refuses the file instead.

In [ ]:
# ============================================================================
# DRILL 1: A scenario file without restart flags
# ============================================================================
broken = [{**s, "turns": [{**{k: v for k, v in t.items() if k != "restart_before"}, "restart": t["restart_before"]}
                          for t in s["turns"]]} for s in SCENARIOS]
silent = run_suite("v1", restarts="none")
restart_slice = [r for r in silent if r["slice"] == "restart"]
print(f"⚠️ With no restarts replayed, v1 passes the restart slice {sum(r['passed'] for r in restart_slice)}/5")
try:
    run_suite("v1", scenarios=broken)
except ValueError as err:
    print(f"❌ validate refuses: {err}")

### 5.3 🐛 Drill 2: the leak fix that keys on the customer alone
Keying sessions by customer ID closes the leak between customers, and merges two sessions of the same customer. The control scenario conv-19 catches it.

In [ ]:
# ============================================================================
# DRILL 2: Customer-only session key
# ============================================================================
class SessionAgentCustomerKey(SessionAgentV2):
    """Drill 2: one conversation per customer, whatever the session ID."""

    def handle(self, customer_id, session_id, message):
        return super().handle(customer_id, "any-session", message)


SESSION_AGENTS["v2-customer-key"] = SessionAgentCustomerKey
rows = run_suite("v2-customer-key")
print(f"⚠️ customer-only key: {sum(r['passed'] for r in rows)}/25")
for r in rows:
    if not r["passed"]:
        print(f"❌ {r['id']}: " + next(c.reason for c in r["results"] if not c.passed))
del SESSION_AGENTS["v2-customer-key"]
print(f"🔧 key (customer, session), as v2 does: {sum(r['passed'] for r in run_suite('v2'))}/25")

---
## 📝 Summary

In this notebook, we learned:

### 1. A restartable harness
- A restart means a new agent object reading the same checkpoint file, while the ledger, like a real backend, stays up.

### 2. State checks
- Outcomes, order in focus, mutation timing, ledger and isolation. v2 passes 25, v1 passes 12, and each failure maps to one bug.

### 3. Tenant isolation
- A session key without the customer hands one customer's order to another. The strict backend blocked the refund, not the leak.

### 4. Restart invariance
- Restarting before every turn changes nothing for v2 and changes all 25 conversations for v1.

### 5. Exact text against state
- Rewording fails 25 of 25 transcripts and 0 state checks. Matching text even passed one conversation whose state was wrong.

### Next Steps
- **`week-10-faults-adversarial-and-handoffs-handson.ipynb`**: timeouts, retries, injected content and handoffs.